> **Ekip paylaşım kopyası:** Kayıtlı çıktılar ve widget durumu dosya boyutunu azaltmak için temizlenmiştir; analiz hücreleri korunur. Veri dosyası Git deposuna dahil değildir. Kurulum, veri yolu ve mevcut veri dosyasını kopyalamadan kullanma seçenekleri için `../README.md` dosyasına bakın. `.venv` kernel'ini seçip Run All çalıştırın. Kaynak projedeki notebook ve çıktılar değiştirilmemiştir.

# DS02 veri yapısının keşfi

Bu notebook `data/N-CMAPSS_DS02-006.h5` dosyasını **salt okunur** açar. Model eğitimi, preprocessing veya dosyaya yazma yapılmaz. Büyük veri dizilerinin yalnızca ilk 10 satırı okunur; değişken adları küçük metadata dizilerinden alınır.

Gereken paketler: `h5py`, `numpy`, `pandas`. Jupyter üzerinden çalıştırmak için ayrıca `ipykernel` gerekir. Kaydedilmiş çıktılar projenin `.venv` ortamındaki gerçek Jupyter çekirdeğiyle üretilmiştir.


In [ ]:
from pathlib import Path
import h5py
import numpy as np
import pandas as pd
from IPython.display import display

# Notebook proje kökünden veya notebooks klasöründen çalıştırılabilir.
# Paylaşım kopyası: veri yolu bilgisayardan bağımsız çözülür.
import os

def resolve_ncmapss_paths(start=None):
    working_directory = Path.cwd().resolve() if start is None else Path(start).resolve()
    project_root = None
    for parent in [working_directory, *working_directory.parents]:
        if parent.name == "n_cmapss" and (parent / "notebooks").is_dir():
            project_root = parent
            break
        candidate = parent / "n_cmapss"
        if (candidate / "notebooks").is_dir():
            project_root = candidate
            break
    if project_root is None:
        raise FileNotFoundError("Notebook'u ortak repo, n_cmapss veya n_cmapss/notebooks klasöründen çalıştırın.")
    external_path = os.environ.get("N_CMAPSS_H5")
    h5_path = Path(external_path).expanduser().resolve() if external_path else project_root / "data" / "N-CMAPSS_DS02-006.h5"
    return project_root, h5_path

project_root, h5_path = resolve_ncmapss_paths()
if not h5_path.is_file():
    raise FileNotFoundError(
        f"Veri dosyası bulunamadı: {h5_path}. HDF5 repoya dahil değildir. "
        "Dosyayı yerel n_cmapss/data/ altında sağlayın veya kernel ortamında "
        "N_CMAPSS_H5 değişkenini mevcut dosyanın tam yoluna ayarlayın. README'ye bakın."
    )

print(f"Dosya: {h5_path}")
print(f"Boyut: {h5_path.stat().st_size:,} bayt")


## 1. Bütün HDF5 nesneleri

HDF5 içinde grup ve dataset farklı nesnelerdir. Aşağıdaki kod bütün alt nesneleri dolaşır; datasetlerin `shape` ve `dtype` özelliklerini okur, içeriklerini belleğe yüklemez. Grupların boyut ve veri tipi alanları uygulanabilir değildir. Her erişimde `with` bloğu dosyayı işlem sonunda kapatır.


In [ ]:
inventory = []
with h5py.File(h5_path, "r") as h5:
    def inspect(name, obj):
        inventory.append({
            "isim": name,
            "nesne_türü": "Dataset" if isinstance(obj, h5py.Dataset) else "Grup",
            "boyut": str(obj.shape) if isinstance(obj, h5py.Dataset) else "—",
            "veri_tipi": str(obj.dtype) if isinstance(obj, h5py.Dataset) else "—",
        })
    h5.visititems(inspect)
inventory_df = pd.DataFrame(inventory)
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(inventory_df)


## 2. Değişken isimleri

Küçük `*_var` dizileri okunur. Bayt dizileri UTF-8 metne çevrilir; her isim kendi sütun indeksiyle gösterilir. Büyük ölçüm dizileri bu aşamada okunmaz.


In [ ]:
variable_keys = ["A_var", "W_var", "X_s_var", "X_v_var", "T_var"]
variable_names = {}
with h5py.File(h5_path, "r") as h5:
    for key in variable_keys:
        raw = np.asarray(h5[key][()]).reshape(-1)
        variable_names[key] = [
            value.decode("utf-8").rstrip("\x00")
            if isinstance(value, (bytes, np.bytes_)) else str(value)
            for value in raw
        ]
        print(f"\n{key} ({len(variable_names[key])} değişken)")
        display(pd.DataFrame({"sütun_indeksi": range(len(variable_names[key])),
                              "değişken": variable_names[key]}))


## 3. Satır sayısı kontrolü ve ilk 10 satır

`A_dev` yardımcı bilgiler (unit, cycle vb.), `W_dev` uçuş koşulları, `X_s_dev` sensör ölçümleri, `X_v_dev` sanal sensörler, `T_dev` bozulma parametreleri ve `Y_dev` RUL değerlerini içerir. Birleştirme öncesinde altı datasetin satır sayıları ve sütun metadatası doğrulanır.

Yalnızca `[:10]` dilimleri alınır. Tablolar aynı orijinal satır indeksiyle yatay birleştirilir; unit veya cycle üzerinden eşleştirme yapılmaz. Bu işlem veri dönüşümü veya preprocessing değildir.


In [ ]:
dev_keys = ["A_dev", "W_dev", "X_s_dev", "X_v_dev", "T_dev", "Y_dev"]
metadata_map = dict(zip(dev_keys[:-1], variable_keys))
with h5py.File(h5_path, "r") as h5:
    row_counts = {key: h5[key].shape[0] for key in dev_keys}
    display(pd.DataFrame.from_dict(row_counts, orient="index", columns=["satır_sayısı"]))
    if len(set(row_counts.values())) != 1:
        raise ValueError(f"dev satır sayıları eşit değil: {row_counts}")
    print(f"Kontrol başarılı: bütün dev datasetlerinde {next(iter(row_counts.values())):,} satır var.")
    sample_count = min(10, next(iter(row_counts.values())))
    frames = []
    for key in dev_keys:
        dataset = h5[key]
        column_count = dataset.shape[1] if dataset.ndim == 2 else 1
        columns = ["RUL"] if key == "Y_dev" else variable_names[metadata_map[key]]
        if dataset.ndim not in (1, 2) or len(columns) != column_count:
            raise ValueError(f"{key}: boyut ve sütun isimleri uyumsuz")
        values = dataset[:sample_count]
        frames.append(pd.DataFrame(values.reshape(sample_count, column_count),
                                   columns=columns, index=pd.RangeIndex(sample_count, name="satır_indeksi")))
    preview = pd.concat(frames, axis=1, verify_integrity=True)
    if not {"unit", "cycle", "RUL"}.issubset(preview.columns):
        raise ValueError("Beklenen unit, cycle veya RUL sütunu eksik")
print(f"Birleşik tablo: {preview.shape[0]} satır, {preview.shape[1]} sütun")
with pd.option_context("display.max_columns", None, "display.width", 240):
    display(preview)


## Kapsam ve bellek kullanımı

Ölçüm verilerinin tamamı okunmadı. Belleğe yalnızca beş değişken adı dizisi ve altı dev datasetinin ilk 10 satırı alındı. HDF5 dosyası bütün erişimlerde `r` modunda açıldı ve kapatıldı. Eğitim, ölçekleme, filtreleme veya başka preprocessing uygulanmadı.
